In [1]:
import os
import random
import shutil
from pathlib import Path

def create_subset_yolo_dataset(
    dataset_path, 
    output_path, 
    fraction=0.2, 
    seed=42
):
    """
    Randomly selects a fraction of images + label pairs from a YOLO dataset.
    Keeps directory structure (train/valid/test) if it exists.
    """
    random.seed(seed)
    dataset_path = Path(dataset_path)
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    total_selected = 0

    yaml_files = list(dataset_path.glob("*.yaml")) + list(dataset_path.glob("*.yml"))
    if yaml_files:
        yaml_src = yaml_files[0]
        yaml_dest = output_path / yaml_src.name
        shutil.copy2(yaml_src, yaml_dest)
        print(f"📄 Copied YAML file: {yaml_src.name}")
    else:
        print("⚠️ No YAML file found in dataset root.")

    for split in ["train", "valid", "test"]:
        img_dir = next(dataset_path.rglob(f"{split}/images"), None)
        lbl_dir = next(dataset_path.rglob(f"{split}/labels"), None)
        if not img_dir or not lbl_dir:
            continue

        # Get only valid pairs
        img_files = [f for f in img_dir.glob("*.*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
        valid_pairs = [f for f in img_files if (lbl_dir / f"{f.stem}.txt").exists()]
        if not valid_pairs:
            continue

        # Randomly sample
        sample_size = max(1, int(len(valid_pairs) * fraction))
        selected = random.sample(valid_pairs, sample_size)

        # Create output dirs
        out_img_dir = output_path / split / "images"
        out_lbl_dir = output_path / split / "labels"
        out_img_dir.mkdir(parents=True, exist_ok=True)
        out_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_file in selected:
            lbl_file = lbl_dir / f"{img_file.stem}.txt"
            shutil.copy2(img_file, out_img_dir / img_file.name)
            shutil.copy2(lbl_file, out_lbl_dir / lbl_file.name)
            total_selected += 1

        print(f"📦 {split}: {len(selected)} / {len(valid_pairs)} samples copied")

    print(f"\n✅ Subset created at: {output_path}")
    print(f"📊 Total selected images: {total_selected}")

create_subset_yolo_dataset(
    dataset_path="mouse.v2i.yolov11",
    output_path="mouse_subset.yolov11",
    fraction=0.1
)

📄 Copied YAML file: data.yaml
📦 train: 531 / 5314 samples copied
📦 valid: 36 / 368 samples copied
📦 test: 26 / 262 samples copied

✅ Subset created at: mouse_subset.yolov11
📊 Total selected images: 593


In [3]:
import os
import shutil
from pathlib import Path
import uuid
import random
import pandas as pd

def remap_yolo_labels(label_path, new_label_id):
    """
    Helper function to overwrite YOLO label IDs with a new one.
    """
    try:
        with open(label_path, "r") as f:
            lines = f.readlines()
        remapped = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) > 0:
                parts[0] = str(new_label_id)  # replace class id
                remapped.append(" ".join(parts))
        with open(label_path, "w") as f:
            f.write("\n".join(remapped))
    except Exception as e:
        print(f"⚠️ Failed to remap {label_path}: {e}")

def create_combined_dataset(datasets, output_dir="combined_dataset", class_map=None):
    """
    Simple function to combine YOLO datasets with unique IDs
    """
    output_path = Path(output_dir)

    # Create directories
    for split in ["train", "valid", "test"]:
        (output_path / split / "images").mkdir(parents=True, exist_ok=True)
        (output_path / split / "labels").mkdir(parents=True, exist_ok=True)

    mapping = []

    for dataset in datasets:
        dataset_path = Path(dataset)
        print(f"📦 Adding dataset: {dataset_path.name}")

        # infer label id based on name
        label_id = None
        for key, value in (class_map or {}).items():
            if key.lower() in dataset_path.name.lower():
                label_id = value
                break

        if label_id is None:
            print(f"⚠️ No class mapping found for {dataset_path.name}, skipping label remap.")
        
        for split in ["train", "valid", "test"]:
            img_dir = next(dataset_path.rglob(f"{split}/images"), None)
            lbl_dir = next(dataset_path.rglob(f"{split}/labels"), None)

            if img_dir and img_dir.exists():
                for img_file in img_dir.glob("*.*"):
                    if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                        unique_id = str(uuid.uuid4())
                        new_img_name = f"{unique_id}{img_file.suffix}"
                        new_lbl_name = f"{unique_id}.txt"

                        # Copy image
                        shutil.copy2(img_file, output_path / split / "images" / new_img_name)

                        # Copy and remap label
                        if lbl_dir and lbl_dir.exists():
                            lbl_file = lbl_dir / f"{img_file.stem}.txt"
                            if lbl_file.exists():
                                new_lbl_path = output_path / split / "labels" / new_lbl_name
                                shutil.copy2(lbl_file, new_lbl_path)
                                if label_id is not None:
                                    remap_yolo_labels(new_lbl_path, label_id)

                        mapping.append({
                            "dataset": dataset_path.name,
                            "new_id": unique_id,
                            "split": split,
                            "class_id": label_id
                        })

    df = pd.DataFrame(mapping)
    df.to_csv(output_path / "file_mapping.csv", index=False)

    print(f"\n✅ Created combined dataset in: {output_dir}")
    print(f"📊 Total images: {len(mapping)}\n\n")
    return output_path


datasets = [
    "my-raccoons.v1i.yolov11",
    "Raccoon.v2-raw.yolov11",
    "Rat.v1i.yolov11",
    "mouse_subset.yolov11" 
]

class_map = {
    "raccoon": 0,
    "rat": 1,
    "mouse": 1
}

raccoon_datasets = [
    "my-raccoons.v1i.yolov11",
    "Raccoon.v2-raw.yolov11",
] 

combined_path = create_combined_dataset(datasets, class_map=class_map)

combined_path_raccoon = create_combined_dataset(raccoon_datasets, output_dir="combined_raccoon_dataset", class_map={"raccoon": 0})

📦 Adding dataset: my-raccoons.v1i.yolov11
📦 Adding dataset: Raccoon.v2-raw.yolov11
📦 Adding dataset: Rat.v1i.yolov11
📦 Adding dataset: mouse_subset.yolov11

✅ Created combined dataset in: combined_dataset
📊 Total images: 1533


📦 Adding dataset: my-raccoons.v1i.yolov11
📦 Adding dataset: Raccoon.v2-raw.yolov11

✅ Created combined dataset in: combined_raccoon_dataset
📊 Total images: 701




In [4]:
import cv2
from pathlib import Path
import tqdm
import shutil

def convert_yolo_dataset_to_grayscale(input_dataset="combined_dataset",
                                      output_dataset="combined_dataset_grayscale"):
    """
    Converts all images in a YOLO-style dataset (train/val/test) to grayscale.
    Copies labels unchanged and writes grayscale images to a new output dataset folder.

    Parameters:
    - input_dataset (str): Path to the original dataset.
    - output_dataset (str): Path to save the grayscale-converted dataset.
    """

    input_path = Path(input_dataset)
    output_path = Path(output_dataset)

    if not input_path.exists():
        raise FileNotFoundError(f"❌ Input dataset not found: {input_dataset}")

    # Create output folder structure
    for split in ["train", "valid", "test"]:
        img_in = input_path / split / "images"
        lbl_in = input_path / split / "labels"
        img_out = output_path / split / "images"
        lbl_out = output_path / split / "labels"

        if not img_in.exists():
            print(f"⚠️ Skipping {split}: no images found.")
            continue

        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)

        print(f"\n🎨 Converting {split} images to grayscale...")

        # Convert all images to grayscale (3-channel)
        for img_path in tqdm.tqdm(list(img_in.glob("*.*"))):
            if img_path.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            gray_3ch = cv2.merge([gray, gray, gray])  # keep 3 channels
            cv2.imwrite(str(img_out / img_path.name), gray_3ch)

        # Copy labels unchanged
        if lbl_in.exists():
            for lbl_file in lbl_in.glob("*.txt"):
                shutil.copy(lbl_file, lbl_out / lbl_file.name)

    print(f"\n✅ Grayscale dataset saved to: {output_path.resolve()}\n\n")

    return output_path

combined_gray_path = convert_yolo_dataset_to_grayscale(
    input_dataset="combined_dataset",
    output_dataset="combined_dataset_grayscale"
)

combined_raccoon_gray_path = convert_yolo_dataset_to_grayscale(
    input_dataset="combined_raccoon_dataset",
    output_dataset="combined_raccoon_dataset_grayscale"
)




🎨 Converting train images to grayscale...


100%|██████████| 1308/1308 [00:08<00:00, 158.37it/s]



🎨 Converting valid images to grayscale...


100%|██████████| 141/141 [00:00<00:00, 173.62it/s]



🎨 Converting test images to grayscale...


...


🎨 Converting valid images to grayscale...


100%|██████████| 58/58 [00:00<00:00, 238.06it/s]



🎨 Converting test images to grayscale...


100%|██████████| 34/34 [00:00<00:00, 242.80it/s]



✅ Grayscale dataset saved to: C:\Users\plebj\Desktop\School\CS5814\combined_raccoon_dataset_grayscale




In [ ]:
def rebalance_train_test_split(input_dir, output_dir="rebalanced_dataset", train_ratio=0.80, valid_ratio=0.10, test_ratio=0.10):
    """
    Rebalance a YOLO dataset into a new train/test split (default 70/20/10).
    Works on the already combined dataset without duplicating counts.
    """
    dataset_path = Path(input_dir)
    output_path = Path(output_dir)

    # Collect all images + labels just once
    all_pairs = []
    for split in ["train", "valid", "test"]:
        img_dir = dataset_path / split / "images"
        lbl_dir = dataset_path / split / "labels"
        if not img_dir.exists():
            continue
        for img_file in img_dir.glob("*.*"):
            if img_file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
                lbl_file = lbl_dir / f"{img_file.stem}.txt"
                if lbl_file.exists():
                    all_pairs.append((img_file, lbl_file))

    print(f"Found {len(all_pairs)} unique image-label pairs in total.")

    # Shuffle
    random.shuffle(all_pairs)

    # Split indices
    n = len(all_pairs)
    train_end = int(n * train_ratio)
    valid_end = train_end + int(n * valid_ratio)

    # Train/Test/Valid split
    train_pairs = all_pairs[:train_end]
    valid_pairs = all_pairs[train_end:valid_end]
    test_pairs  = all_pairs[valid_end:]

    # Reset rebalanced folder
    if output_path.exists():
        shutil.rmtree(output_path)

    for split in ["train", "valid", "test"]:
        (output_path / split / "images").mkdir(parents=True, exist_ok=True)
        (output_path / split / "labels").mkdir(parents=True, exist_ok=True)

    # Copy files into new dataset
    def copy_pairs(pairs, split):
        for img_file, lbl_file in pairs:
            shutil.copy2(img_file, output_path / split / "images" / img_file.name)
            shutil.copy2(lbl_file, output_path / split / "labels" / lbl_file.name)

    copy_pairs(train_pairs, "train")
    copy_pairs(valid_pairs, "valid")
    copy_pairs(test_pairs, "test")

    print(f"Created rebalanced dataset at: {output_path}")
    print(f"{len(train_pairs)} train, {len(valid_pairs)} valid, {len(test_pairs)} test \n\n")
    return output_path

rebalance_train_test_split(input_dir="combined_dataset_grayscale")
rebalance_train_test_split(input_dir="combined_raccoon_dataset_grayscale", output_dir="rebalanced_raccoon_dataset")

Found 1533 unique image-label pairs in total.
Created rebalanced dataset at: rebalanced_dataset
1226 train, 153 valid, 154 test
Found 701 unique image-label pairs in total.
Created rebalanced dataset at: rebalanced_raccoon_dataset
560 train, 70 valid, 71 test


WindowsPath('rebalanced_raccoon_dataset')

In [ ]:
import os
from pathlib import Path

def count_yolo_dataset(dataset_path):
    dataset_path = Path(dataset_path)
    splits = ["train", "valid", "test"]

    counts = {}
    for split in splits:
        # Look for split folders anywhere inside dataset_path
        img_dirs = list(dataset_path.rglob(f"{split}/images"))
        lbl_dirs = list(dataset_path.rglob(f"{split}/labels"))

        img_count = sum(len(list(d.glob("*.jpg"))) + len(list(d.glob("*.png"))) for d in img_dirs)
        lbl_count = sum(len(list(d.glob("*.txt"))) for d in lbl_dirs)

        # Only include split if any images or labels were found
        if img_count > 0 or lbl_count > 0:
            counts[split] = {
                "images": img_count,
                "labels": lbl_count
            }

    return counts

datasets = [
    "my-raccoons.v1i.yolov11",
    "Raccoon.v2-raw.yolov11",
    "Rat.v1i.yolov11",
    "mouse_subset.yolov11",
    "rebalanced_dataset",
    "rebalanced_raccoon_dataset"
]

for ds in datasets:
    print(f"Dataset: {ds}")
    try:
        counts = count_yolo_dataset(ds)
        if counts:
            for split, stats in counts.items():
                print(f"  {split}: {stats['images']} images, {stats['labels']} labels")
        else:
            print("  No images/labels found in any split.")
    except Exception as e:
        print(f"  Error reading {ds}: {e}")


Dataset: my-raccoons.v1i.yolov11
  train: 459 images, 459 labels
  valid: 29 images, 29 labels
  test: 17 images, 17 labels
Dataset: Raccoon.v2-raw.yolov11
  train: 150 images, 150 labels
  valid: 29 images, 29 labels
  test: 17 images, 17 labels
Dataset: Rat.v1i.yolov11
  train: 168 images, 168 labels
  valid: 47 images, 47 labels
  test: 24 images, 24 labels
Dataset: mouse_subset.yolov11
  train: 531 images, 531 labels
  valid: 36 images, 36 labels
  test: 26 images, 26 labels
Dataset: mouse.v2i.yolov11
  train: 5314 images, 5314 labels
  valid: 368 images, 368 labels
  test: 262 images, 262 labels
Dataset: rebalanced_dataset
  train: 1226 images, 1226 labels
  valid: 153 images, 153 labels
  test: 154 images, 154 labels
Dataset: rebalanced_raccoon_dataset
  train: 560 images, 560 labels
  valid: 70 images, 70 labels
  test: 71 images, 71 labels
